## Graph 1: Relationship Between Student Procrastination, AI Usage, and Reasons for Using AI

**Graph Type:** Sankey Diagram

**Description:** 

**Hypothesis:** 

**Takeaway:** 



In [2]:
import pandas as pd

# Load the original dataset
df = pd.read_csv("../../data/uncleaned_data.csv")

# Keep the original dataset unchanged
df_raw = df.copy()

# Create analysis dataframe
df_sankeyAnalysis = pd.DataFrame()

df_sankeyAnalysis["ai_days"] = pd.to_numeric(df[next(col for col in df.columns if "on how many days did you use AI tools for schoolwork" in col)],errors="coerce")

df_sankeyAnalysis["ai_minutes"] = pd.to_numeric(df[next(col for col in df.columns if "Total minutes:" in col)],errors="coerce")

df_sankeyAnalysis["main_reason"] = df[next(col for col in df.columns
if "main reason for using AI" in col)]

df_sankeyAnalysis["reliance_after_procrastinating"] = df[next(col for col in df.columns
if "I find myself relying on AI more after I procrastinate" in col)]

df_sankeyAnalysis["homework_timing"] = df[next(col for col in df.columns
if "How far in advance do you typically begin working" in col
and "[Homework]" in col)]

df_sankeyAnalysis["coursework_stress"] = pd.to_numeric(df[next(col for col in df.columns
if "[Stress about coursework]" in col)],errors="coerce")

# Clean text columns
text_columns = ["main_reason","reliance_after_procrastinating","homework_timing"]

for col in text_columns:
    df_sankeyAnalysis[col] = df_sankeyAnalysis[col].str.strip()


In [3]:
import plotly.graph_objects as go

sankey_data = df_sankeyAnalysis[["reliance_after_procrastinating","ai_days","main_reason"]].copy()


sankey_data = sankey_data.dropna(subset=["reliance_after_procrastinating","ai_days","main_reason"]).copy()


def categorize_days(days):
    if days == 0:
        return "No AI use"
    elif days <= 2:
        return "1 to 2 days"
    elif days <= 4:
        return "3 to 4 days"
    else:
        return "5 to 7 days"

sankey_data["ai_usage"] = sankey_data["ai_days"].apply(categorize_days)


sankey_data["main_reason"] = (sankey_data["main_reason"].astype(str).str.replace(r"Generating answers,\s*text,\s*or code","Generating answers text or code",regex=True,case=False))


sankey_data["main_reason"] = (sankey_data["main_reason"].str.split(r"\s*,\s*"))

sankey_data = sankey_data.explode("main_reason")


sankey_data["main_reason"] = (sankey_data["main_reason"].str.replace("Generating answers text or code", "Generating answers, text, or code",regex=False).str.strip())


conflicts = (sankey_data["main_reason"].str.contains("Did not use AI",case=False,na=False) & (sankey_data["ai_days"] > 0))

print("Conflicting responses:", conflicts.sum())

sankey_data.loc[conflicts, "main_reason"] = "Conflicting responses"


print("Total respondent-reason paths:", len(sankey_data))
print("Unique students:", sankey_data.index.nunique())


reliance_categories = ["Never","Rarely","Sometimes","Often","Always"]

usage_categories = ["No AI use","1 to 2 days","3 to 4 days","5 to 7 days"]

reason_categories = sorted(sankey_data["main_reason"].unique())


nodes = (["Reliance: " + x for x in reliance_categories] + ["AI Usage: " + x for x in usage_categories] + ["Reason: " + x for x in reason_categories])

node_ids = {label: i for i, label in enumerate(nodes)}

sources = []
targets = []
values = []


def add_connections(source_col,target_col,source_prefix,target_prefix):
    grouped = (sankey_data.groupby([source_col, target_col]).size().reset_index(name="count"))

    for _, row in grouped.iterrows():
        source = source_prefix + str(row[source_col])
        target = target_prefix + str(row[target_col])
        sources.append(node_ids[source])
        targets.append(node_ids[target])
        values.append(row["count"])


add_connections("reliance_after_procrastinating","ai_usage","Reliance: ","AI Usage: ")


add_connections("ai_usage","main_reason","AI Usage: ","Reason: ")


fig = go.Figure(data=[go.Sankey(arrangement="snap",node=dict(pad=25,thickness=22,line=dict(color="black",width=0.5),label=nodes),link=dict(source=sources,target=targets,value=values))])

fig.update_layout(title_text="Student Procrastination, AI Usage, and Individual Motivations",font_size=12,height=800,width=1200)

fig.show()


Conflicting responses: 2
Total respondent-reason paths: 180
Unique students: 78


In [4]:
from scipy.stats import spearmanr
import pandas as pd

correlation_data = df_sankeyAnalysis[["reliance_after_procrastinating", "ai_days"]].copy()

reliance_mapping = {"Never": 1,"Rarely": 2,"Sometimes": 3,"Often": 4,"Always": 5}

correlation_data["reliance_score"] = (correlation_data["reliance_after_procrastinating"].map(reliance_mapping))

correlation_data = correlation_data.dropna(subset=["reliance_score", "ai_days"])

correlation, p_value = spearmanr(correlation_data["reliance_score"],correlation_data["ai_days"])

print("Number of students:", len(correlation_data))
print("Spearman correlation:", correlation)
print("P-value:", p_value)


Number of students: 79
Spearman correlation: 0.3999209103290754
P-value: 0.0002606616483364887




What we suspect

We think students who report relying on AI more after procrastinating are also more likely to use AI frequently throughout the week for schoolwork

Our analysis

Converted the responses about relying on AI after procrastinating into numerical values from 1 to 5. Used the number of days each student reported using AI for schoolwork during the past week of 0-7 days. We used Spearman's correlation to test because the responses are ordered categories.

Null Hypothesis (H₀): There is no correlation between how often students rely on AI after procrastinating and how many days they use AI for schoolwork

Alternative Hypothesis (H₁): There is a correlation between the two variables.


Result and conclusion:

Findings

Valid paired responses: 79

Spearman correlation coefficient: 0.3999

P-value: 0.000261

Significance level: 0.05

The correlation is positive and statistically significant.

Conclusion

Since the p-value is smaller than 0.05 we reject the null hypothesis. The results indicate a moderate positive correlation between how often students rely on AI after procrastinating and how many days they use AI for schoolwork. This suggests that students who report relying on AI more after procrastinating also tend to use AI more frequently throughout the week. But this does not mean that procrastination causes students to use AI more frequently